# 交割单分析：独立复核记录

本笔记核对 2026-07-01 至 2026-10-17 的已提取交割单，估值截至 10 月 16 日收盘。
主样本为 **158 轮成本已知、已平仓的二级市场交易**，交易净损益 **−48,511.84 元**，61 胜；两轮新股中签独立列示。

口径与边界：交易净损益＝净卖出收入−含费买入成本；分红、税和利息独立列示，故这里的胜负数与含分红的账务汇总可能不同。每轮收益率分母为累计买入成本，**不是账户收益率**。买点只使用前一交易日及更早的日K和实际首笔买入价；买入日复权因子仅用于统一价格单位。事件时间不代表已经核验的成交时刻。

输入全部使用同目录相对路径；不联网、不修改输入、不重建全部报告图表。除 Python 标准库外无需分析依赖。当前环境没有 nbformat、nbclient 或 Jupyter 内核，五个单元由标准库执行器在一个干净命名空间按顺序实际执行并保存输出；**不是 nbclient 内核执行记录**。Notebook 外观未另作渲染检查。

## 1. 输入与复现

将本文件与 `episodes.json`、`analysis.json`、`market/daily_bars.json` 保持相同相对结构，从笔记所在目录运行。哈希用来识别本次输入快照。

In [ ]:
from pathlib import Path
from decimal import Decimal
from collections import defaultdict, Counter
import json, hashlib, statistics, math

ROOT = Path.cwd()
input_names = ['episodes.json', 'analysis.json', 'market/daily_bars.json']
for name in input_names:
    assert (ROOT / name).is_file(), f'请将工作目录设为本笔记所在目录：缺少 {name}'
episodes, analysis, bars = [json.loads((ROOT / name).read_text()) for name in input_names]
def money_sum(values):
    return float(sum((Decimal(str(v)) for v in values), Decimal('0')))
def close(a, b, tol=0.011):
    assert abs(a-b) <= tol, (a, b)
for name in input_names:
    print(name, 'sha256=' + hashlib.sha256((ROOT / name).read_bytes()).hexdigest())
print(f'读取：{len(episodes)} 轮、{len(bars):,} 根日K；行情截止 {analysis["scope"]["marketAsOf"]}。')

## 2. 主交易损益、新股与排除项

二级市场闭合轮次从逐笔现金发生额重新加总，不直接加总报告中的分析结果。新股缴款成本沿用已核对的分配成本，单独显示。期初成本未知的一轮不冒充零成本盈利；未平仓的一轮另作市值估算。

In [ ]:
secondary = [e for e in episodes if e['status']=='closed' and e['initial_cost_known'] and e['origin']=='secondary']
ipo = [e for e in episodes if e['status']=='closed' and e['initial_cost_known'] and e['origin']=='ipo']
raw_pnl, raw_cost = {}, {}
for e in secondary:
    events = e['event_details']
    buy_cost = -money_sum(x['net'] for x in events if x['kind']=='证券买入')
    sell_cash = money_sum(x['net'] for x in events if x['kind']=='证券卖出')
    pnl = round(sell_cash-buy_cost, 2)
    close(buy_cost, e['total_buy_cost']); close(sell_cash, e['sell_net_proceeds'])
    close(pnl, e['trade_pnl'])
    raw_pnl[e['episode_id']], raw_cost[e['episode_id']] = pnl, buy_cost
total_pnl = money_sum(raw_pnl.values())
wins = sum(v>0 for v in raw_pnl.values())
assert len(secondary)==158 and wins==61 and len(ipo)==2
close(total_pnl, -48511.84)
close(total_pnl, analysis['summary']['closed_secondary']['trade_pnl'])
ipo_pnl = money_sum(e['sell_net_proceeds']-e['total_buy_cost'] for e in ipo)
close(ipo_pnl, 47547.78)
assert sum(e['status']=='open' for e in episodes)==1
assert sum(not e['initial_cost_known'] for e in episodes)==1
print(f'二级市场：{len(secondary)} 轮，{wins} 胜，{len(secondary)-wins} 负，胜率 {wins/len(secondary):.2%}，交易净损益 {total_pnl:,.2f} 元。')
print(f'新股中签：{len(ipo)} 轮，净损益 {ipo_pnl:,.2f} 元；已平仓合计 {total_pnl+ipo_pnl:,.2f} 元。')
print('已确认：1 轮期初成本未知、1 轮未平仓均从主样本排除。')

## 3. 逐交易日期覆盖与可用的历史信息

逐笔买卖须有对应当日日K，成交价须落在当天未复权高低价之间。新股入账等非买卖事件不要求入账当天已经上市。历史均线采用买入当日价格单位，使用买入日前已完成的日K。

In [ ]:
def ticker(code):
    return code + ('.SH' if code[0] in '569' else '.SZ' if code[0] in '0123' else '.BJ')
bar_map = {(b['ticker'], b['date']): b for b in bars}
assert len(bar_map)==len(bars) and all(b.get('adj_factor') for b in bars)
by_ticker = defaultdict(list)
for b in bars: by_ticker[b['ticker']].append(b)
for series in by_ticker.values(): series.sort(key=lambda b:b['date'])
trades = {}
for e in episodes:
    for event in e['event_details']:
        if event['kind'] in ['证券买入', '证券卖出']:
            assert event['row_id'] not in trades
            trades[event['row_id']] = (ticker(e['code']), event)
assert len(trades)==462
for ts, event in trades.values():
    b = bar_map[(ts,event['date'])]
    assert b['low']-0.011 <= event['price'] <= b['high']+0.011
for date in {event['date'] for ts,event in trades.values()}:
    for ts in ['000001.SH','000300.SH','399006.SZ']: assert (ts,date) in bar_map
def features(ts, date, price=None):
    prior = [b for b in by_ticker[ts] if b['date']<date]
    if len(prior)<25: return {'history':len(prior)}
    ref = bar_map.get((ts,date),prior[-1])['adj_factor']
    closes = [b['close']*b['adj_factor']/ref for b in prior]
    ma5,ma20 = statistics.mean(closes[-5:]),statistics.mean(closes[-20:])
    result = {'history':len(prior),'prev':closes[-1],'ma5':ma5,'ma20':ma20,
              'slope':ma20/statistics.mean(closes[-25:-5])-1}
    if price is not None: result.update(chase=(price/closes[-1]-1)*100,stretch=(price/ma5-1)*100)
    return result
def regime(f):
    if f['history']<25:return '行情不足'
    if f['prev']>f['ma20'] and f['slope']>0:return '大盘上升趋势'
    if f['prev']<f['ma20'] and f['slope']<0:return '大盘下降趋势'
    return '大盘震荡过渡'
def entry(f,chase=3,stretch=8):
    if f['history']<25:return '历史不足或行情缺失'
    if f['chase']>=chase or f['stretch']>=stretch:return '明显追涨或远离5日线'
    up = f['prev']>f['ma20'] and f['slope']>0
    if up and abs(f['stretch'])<=3:return '上升趋势且靠近5日线'
    if up:return '上升趋势其他买入'
    if f['prev']<f['ma20'] and f['slope']<0:return '下跌趋势中买入'
    return '震荡或反弹中买入'
feature_map = {e['episode_id']:features(ticker(e['code']),e['first_buy_date'],e['first_buy_price']) for e in secondary}
market_map = {e['episode_id']:regime(features('000001.SH',e['first_buy_date'])) for e in secondary}
growth_map = {e['episode_id']:regime(features('399006.SZ',e['first_buy_date'])) for e in secondary}
print(f'{len(trades)} 条买卖全部有当日日K，全部成交价在当日高低价内；三类指数在全部交易日期覆盖完整。')
print(f'{len(bars):,} 根日K复权因子完整；主样本历史不足25根的轮次 {sum(f["history"]<25 for f in feature_map.values())}。')

## 4. 分组与整体对账

不同维度各自穷尽同一主样本，但不同维度之间不能相加。独立重建买点和大盘分类，并核对各组轮数及损益。

In [ ]:
index_dates = [b['date'] for b in by_ticker['000001.SH']]
def hold_days(e):return sum(e['first_buy_date']<d<=e['last_sell_date'] for d in index_dates)
definitions = {
 'entry_patterns':lambda e:entry(feature_map[e['episode_id']]),
 'market_regimes':lambda e:market_map[e['episode_id']],
 'growth_market_regimes':lambda e:growth_map[e['episode_id']],
 'behavior':lambda e:'曾在持仓均价下方加仓' if e['add_below_cost_count'] else '加仓但未低于持仓均价' if e['add_count'] else '全程不加仓',
 'holding':lambda e:'持有1个交易日以内' if hold_days(e)<=1 else '持有2至3个交易日' if hold_days(e)<=3 else '持有4个交易日以上',
 'size':lambda e:'最高成本敞口不足1万元' if e['max_wa_cost_exposure']<10000 else '最高成本敞口1至3万元' if e['max_wa_cost_exposure']<30000 else '最高成本敞口3万元以上',
 'asset':lambda e:'ETF' if e['asset_type']=='ETF' else '股票',
 'combinations':lambda e:entry(feature_map[e['episode_id']])+' × '+market_map[e['episode_id']],
}
for dimension,group_key in definitions.items():
    actual = defaultdict(list)
    for e in secondary:actual[group_key(e)].append(e)
    reported = {g['group']:g for g in analysis[dimension]}
    assert set(actual)==set(reported), dimension
    for name,members in actual.items():
        assert len(members)==reported[name]['count'], (dimension,name)
        close(money_sum(raw_pnl[e['episode_id']] for e in members),reported[name]['pnl'])
    assert sum(g['count'] for g in reported.values())==158
    close(money_sum(g['pnl'] for g in reported.values()),total_pnl)
    print(f'{dimension}: {len(actual)} 组；158 轮、{total_pnl:,.2f} 元，逐组和合计均吻合。')

## 5. 未平仓市值与盈利候选的敏感性

未平仓估值不扣未来卖出费用。候选组合为“明显追涨或远离5日线 × 大盘震荡过渡”；剔除8月按首次买入月份，更换创业板基准会重新选择样本。

In [ ]:
open_e = next(e for e in episodes if e['status']=='open')
last = max((b for b in by_ticker[ticker(open_e['code'])] if b['date']<=analysis['scope']['marketAsOf']),key=lambda b:b['date'])
mv = open_e['remaining_quantity']*last['close']
fifo_unrealized = mv-open_e['remaining_cost_fifo']
fifo_total = fifo_unrealized+open_e['fifo_realized_pnl']
wa_total = mv-open_e['remaining_cost_wa']+open_e['wa_realized_pnl']
cash_mtm = mv+open_e['sell_net_proceeds']-open_e['total_buy_cost']
close(fifo_total,wa_total);close(fifo_total,cash_mtm);close(fifo_total,171.99)
reported_open = analysis['open_positions'][0]
close(mv,reported_open['marketValue']);close(fifo_unrealized,reported_open['unrealizedPnlFifo'])
close(fifo_total,reported_open['totalEpisodeMtm'])
print(f'未平仓：{open_e["code"]}，{open_e["remaining_quantity"]:,.0f} 股 × {last["close"]:.2f} 元，市值 {mv:,.2f} 元。')
print(f'FIFO浮盈 {fifo_unrealized:,.2f} 元；包含本轮已实现损益后的MTM {fifo_total:,.2f} 元，移动均价法及现金法均一致。')
def candidate(chase=3,stretch=8,market=market_map):
    return [e for e in secondary if entry(feature_map[e['episode_id']],chase,stretch)=='明显追涨或远离5日线' and market[e['episode_id']]=='大盘震荡过渡']
def metrics(members):
    p = [raw_pnl[e['episode_id']] for e in members]
    returns = [raw_pnl[e['episode_id']]/raw_cost[e['episode_id']]*100 for e in members]
    return len(members),round(money_sum(p),2),statistics.mean(returns),statistics.median(returns)
base = candidate()
checks = [
 ('原定义3%/8%',base,37,6425.26),
 ('阈值2%/6%',candidate(2,6),39,6817.83),
 ('阈值5%/10%',candidate(5,10),27,3521.63),
 ('剔除金健米业',[e for e in base if e['code']!='600127'],34,4089.53),
 ('剔除8月',[e for e in base if e['first_buy_date'][:7]!='2026-08'],10,30.03),
 ('两者均剔除',[e for e in base if e['code']!='600127' and e['first_buy_date'][:7]!='2026-08'],8,-2174.63),
 ('创业板基准',candidate(market=growth_map),50,-10584.84),
]
for label,members,expected_count,expected_pnl in checks:
    n,pnl,mean_ret,median_ret = metrics(members)
    assert n==expected_count;close(pnl,expected_pnl)
    print(f'{label}: {n}轮 | {pnl:,.2f}元 | 均收益 {mean_ret:+.4f}% | 中位 {median_ret:+.4f}%')
august_pnl = money_sum(raw_pnl[e['episode_id']] for e in base if e['first_buy_date'][:7]=='2026-08')
print(f'8月贡献原候选净利润 {august_pnl/metrics(base)[1]:.2%}。全部独立复核通过。')

## 结论与验证边界

所有计算断言通过。主样本不是稳定盈利状态；候选组合的原始净金额为正，但等权单轮均收益接近零、利润集中在8月，更换市场基准或同时排除8月与金健米业后转负，不能将其描述为已经证实的稳定模式。

本笔记复核的是已保存的归集轮次、行情及报告数值，未重新登录券商、逐页提取交割单，亦未在此重新构造FIFO配对。它复算现金口径、分组和市值，不能替代原始抽取完整性检查或证明未来交易收益。